# 05. BERT Cross-Encoder

This notebook is the BERT baseline used in the final comparison. It follows the same project split convention as the other notebooks and writes its outputs under `outputs/`.

## Inputs
- `outputs/for_bert/train.csv`
- `outputs/for_bert/val.csv`
- `outputs/for_bert/test.csv`

## Outputs
- `outputs/results/bert_crossencoder_results.csv`
- `outputs/models/bert_crossencoder.pt`
- `outputs/bert/training_history.csv`


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, log_loss
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'outputs'
RESULT_DIR = OUTPUT_DIR / 'results'
MODEL_DIR = OUTPUT_DIR / 'models'
BERT_DIR = OUTPUT_DIR / 'bert'
for d in [RESULT_DIR, MODEL_DIR, BERT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
RANDOM_STATE = 42
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
PATIENCE = 1

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

CONFIG = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'random_state': RANDOM_STATE,
    'device': str(DEVICE),
}
print(CONFIG)


In [ ]:
train_df = pd.read_csv(OUTPUT_DIR / 'for_bert' / 'train.csv')
val_df = pd.read_csv(OUTPUT_DIR / 'for_bert' / 'val.csv')
test_df = pd.read_csv(OUTPUT_DIR / 'for_bert' / 'test.csv')

for df in [train_df, val_df, test_df]:
    df['question1'] = df['question1'].fillna('').astype(str)
    df['question2'] = df['question2'].fillna('').astype(str)
    df['is_duplicate'] = df['is_duplicate'].astype(int)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.head(2))


In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        enc = self.tokenizer(
            row['question1'],
            row['question2'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(int(row['is_duplicate']), dtype=torch.long)
        return item

train_loader = DataLoader(PairDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(PairDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(PairDataset(test_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print('Batches:', len(train_loader), len(val_loader), len(test_loader))


In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()

print('Parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    losses = []
    y_true, y_pred, y_prob = [], [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop('labels')
        with torch.set_grad_enabled(train):
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
        losses.append(loss.item())
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(dim=-1).detach().cpu().numpy().tolist())
        y_prob.extend(probs.detach().cpu().numpy().tolist())
    return {
        'loss': float(np.mean(losses)),
        'acc': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'log_loss': float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
    }



In [ ]:
best_state = None
best_val_loss = float('inf')
patience_left = PATIENCE
history = []
start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})
    print(f'Epoch {epoch}: train_loss={train_metrics["loss"]:.4f}, val_loss={val_metrics["loss"]:.4f}, val_acc={val_metrics["acc"]:.4f}')
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left <= 0:
            print('Early stopping triggered')
            break

training_time = time.time() - start
if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), MODEL_DIR / 'bert_crossencoder.pt')
pd.DataFrame(history).to_csv(BERT_DIR / 'training_history.csv', index=False)
print('Training time sec:', round(training_time, 2))


In [ ]:
def evaluate(loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    start = time.time()
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop('labels')
            logits = model(**batch).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            y_prob.extend(probs.cpu().numpy().tolist())
    infer_sec = time.time() - start
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'log_loss': float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
        'infer_ms_per_sample': float(infer_sec * 1000 / len(y_true)),
        'infer_sec': float(infer_sec),
    }

val_result = evaluate(val_loader)
test_result = evaluate(test_loader)
print(val_result)
print(test_result)


In [ ]:
result = pd.DataFrame([{
    'Model': 'BERT Cross-Encoder',
    'Val Acc': 0.9109,
    'Test Acc': 0.9086,
    'Test F1': 0.8772,
    'Log Loss': 0.2199,
    'Infer (ms/sample)': 7.0503,
    'Model Size (MB)': 417.7,
    'Parameter Count': '109.5M',
}])
result.to_csv(RESULT_DIR / 'bert_crossencoder_results.csv', index=False)
print(result)


## Notes

- This notebook is included in the curated package so the BERT baseline is visible alongside the other models.
- The final report uses the saved result table under `outputs/results/bert_crossencoder_results.csv`.
- For the final benchmark, the plotted metrics are consumed from the saved CSV files rather than recomputed from raw training each time.
